In [ ]:
"""
sandbox_lite.ipynb

A sandbox to develope a lighter version of the code.

Author: Stellina X. Ao
Created: 2026-05-29
Last Modified: 2026-05-29
Python Version: 3.11.14
"""


import scienceplots  # noqa: F401
import shutup
import matplotlib.pyplot as plt

%load_ext autoreload
%autoreload 2

# pretty plots
plt.style.use(["nature"])
plt.rcParams["figure.dpi"] = 200
%matplotlib widget
%config InlineBackend.print_figure_kwargs = {'bbox_inches':None}

# suppress warnings :-)
shutup.please()

In [ ]:
subj_id = "MR82"
sess_id = "20251027_152036"

In [ ]:
from core.data import load_sess

(
    spike_times,
    trial_data,
    psths,
    session_data,
    regions,
) = load_sess(
    subj_id=subj_id,
    sess_id=sess_id,
    tpre=0.5,
    tpost=1,
    alignment="choice",
    binwidth_ms=25,
    thresh=1,
)

In [ ]:
trial_start = session_data["events"]["event_timestamps"][0]
movie_frame = session_data["events"]["event_timestamps"][1]

In [ ]:
trial_data[-2:]

In [ ]:
# add in the last trial
trial_data.columns

In [ ]:
import numpy as np

movements = np.load(
    "/Users/stellina/Desktop/ChurchlandLab/data-np/MR82/20251027_152036/MR82_DynamicForaging_20251027_152036_cam1_run000_00000000.npy",
    allow_pickle=True,
).item()

In [ ]:
plt.figure()
plt.scatter(
    movements["SVT"][0],
    movements["SVT"][1],
    s=0.5,
    alpha=0.5,
    c=np.arange(movements["SVT"].shape[1]),
)
plt.show()

In [ ]:
trial_data["outcome_time"].iloc[-1]

In [ ]:
import numpy as np


def get_movement_trial_idxs(trial_start, trial_data, movie_frame):
    frame_trial_idxs = np.zeros((len(trial_start), 2), dtype=np.int32)

    trial_start = np.append(
        trial_start, trial_start[-1] + trial_data["outcome_time"].iloc[-1]
    )  # TODO

    for trial_i in range(frame_trial_idxs.shape[0]):
        start_idx = np.searchsorted(movie_frame, trial_start[trial_i])
        end_idx = np.searchsorted(movie_frame, trial_start[trial_i + 1])

        frame_trial_idxs[trial_i] = (start_idx, end_idx)

    return frame_trial_idxs


frame_trial_idxs = get_movement_trial_idxs(trial_start, trial_data, movie_frame)

In [ ]:
import pandas as pd

svd_tavg = np.zeros((len(frame_trial_idxs), 200))
for i, (start, end) in enumerate(frame_trial_idxs):
    svd_tavg[i] = movements["SVT"][:, start:end].mean(axis=1)

svd_df = pd.DataFrame(svd_tavg, columns=[f"SVD_{i}" for i in range(200)])

In [ ]:
plt.figure()
plt.scatter(
    movements["SVT"][0], movements["SVT"][1], c=range(movements["SVT"].shape[1])
)
plt.show()

In [ ]:
plt.figure()
plt.scatter(svd_df[0], svd_df[1], c=np.arange(262))
plt.show()

In [ ]:
trial_data = trial_data.join(svd_df)

In [ ]:
trial_data

In [ ]:
svd_df[248:253]

In [ ]:
svd_df